# Stpm Workflow
Run the complete paper analysis and render its figures and Tables 6–10.

Both notebooks use `crime.downloader`, `importer`, `generator`, `warehouse`,
`stpm`, `validation`, `visualizer` and `exporter`. The notebooks use the pinned SSP spatial study and internal data-quality checks.

[Colab form fields](https://colab.research.google.com/notebooks/forms.ipynb)
· [Package architecture](../docs/PACKAGE_ARCHITECTURE.md)


## 1. Workspace and package
Run cells in order. Colab form fields are editable parameters; in Jupyter, edit
these same Python assignments. The setup cell installs the public package from GitHub, and the quick-start
cell downloads the public pinned data release automatically. No wheel or raw-file upload is required.

Colab storage and a temporary PostGIS database disappear when the runtime ends.
Choose a workspace on mounted Drive, or download the final results archive.
Installing the package can require restarting the runtime if Colab has already
imported incompatible versions; rerun setup after restarting.


In [ ]:
# @title Workspace and installation
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = "" # @param {type:"string"}
STUDY_DATA_DIRECTORY = "" # @param {type:"string"}
PACKAGE_SOURCE = "git+https://github.com/rsushizawa/campinas-crime-stpm.git" # @param {type:"string"}
INSTALL_PACKAGE = True # @param {type:"boolean"}

IN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
ROOT = Path(PROJECT_ROOT).expanduser() if PROJECT_ROOT else Path.cwd() / "crime-workspace"
ROOT.mkdir(parents=True, exist_ok=True)
os.environ["CRIME_PROJECT_ROOT"] = str(ROOT.resolve())
if STUDY_DATA_DIRECTORY:
    os.environ["CRIME_DATA_DIR"] = str(Path(STUDY_DATA_DIRECTORY).expanduser())
if INSTALL_PACKAGE and PACKAGE_SOURCE and importlib.util.find_spec("crime") is None:
    requirement = ("campinas-crime-stpm[etl,db] @ " + PACKAGE_SOURCE
                   if PACKAGE_SOURCE.startswith("git+") else PACKAGE_SOURCE + "[etl,db]")
    subprocess.run([sys.executable, "-m", "pip", "install", requirement], check=True)
if importlib.util.find_spec("crime") is None:
    raise RuntimeError("Package installation did not complete; rerun this setup cell after the pip step")
from IPython.display import Markdown, display

import crime

print("Package:", crime.__version__, "| workspace:", ROOT)


## Quick start
The one-call workflow provisions the selected data profile, builds the treated dataset and opens the starting cube. Run this cell for the VoxCity-style path; the later cells run the paper analyses and publication exports.


In [ ]:
# @title One-call study workflow { display-mode: "form" }
DATA_PROFILE = "snapshot" # @param ["snapshot", "live", "hybrid"]
DATA_MODE = "auto" # @param ["local", "download", "auto"]
AUTO_UPDATE_DATA = False # @param {type:"boolean"}
from crime import CrimeStudy
from crime.config import INPUTS_DIR

study = CrimeStudy(data_profile=DATA_PROFILE, data_mode=DATA_MODE,
                    data_directory=Path(STUDY_DATA_DIRECTORY).expanduser() if STUDY_DATA_DIRECTORY else INPUTS_DIR,
                    auto_update_data=AUTO_UPDATE_DATA)
result = study.build()
display(result.explore())


## 2. Sources and study parameters
**SSP** provides the pinned Campinas 2022–2025 report-level spatial study.
The study uses the pinned SSP report-level inputs and the configured snapshot,
live, or hybrid data profile. Internal checksum, schema, treatment, coordinate,
and tract-assignment checks are retained.


In [ ]:
# @title Study data parameters
NOTEBOOK_NAME = "stpm_exploration"
NOTEBOOK_NAME = "stpm_workflow"
REPORT_SOURCE = "ssp"
DATA_PROFILE = "snapshot" # @param ["snapshot", "live", "hybrid"]
DATA_MODE = "local" # @param ["local", "download", "auto"]
DATA_RELEASE_URL = "" # @param {type:"string"}
AUTO_UPDATE_DATA = False # @param {type:"boolean"}
LIVE_SOURCE_URLS_JSON = "{}" # @param {type:"string"}

from datetime import datetime, timezone

RUN_DIR = ROOT / "outputs/guided" / (NOTEBOOK_NAME + "_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Results:", RUN_DIR)


## 3. Acquire and treat the spatial-study inputs
Existing staged data can be reused. Otherwise the package applies its pinned
acquisition, cleaning and tract-assignment rules. Missing inputs can be downloaded
from an explicit release URL exposing the paths in the packaged `campinas-2022-2025.sha256` manifest.
The package verifies all checksums and never substitutes aggregate counts for
individual reports. If inputs are private, mount/upload them into STUDY_DATA_DIRECTORY before importing the package.


In [ ]:
# @title Prepare SSP reports, IBGE tracts and CNEFE addresses
import json

import pandas as pd

from crime.config import INPUTS_DIR, STAGING_DIR
from crime.generator import generate_dataset, load_dataset

dataset = None
if REPORT_SOURCE == "ssp":
    live_sources = json.loads(LIVE_SOURCE_URLS_JSON or "{}")
    if DATA_PROFILE == "live":
        from crime.downloader.live import download_live_inputs
        live_data = download_live_inputs(
            INPUTS_DIR, source_urls=live_sources, overwrite=(DATA_MODE == "download" or AUTO_UPDATE_DATA),
            require_all=True)
        display({"data_profile": "live", "retrieved_at": live_data.retrieved_at})
    elif DATA_MODE != "download" and (STAGING_DIR / "units/bulletins.parquet").is_file():
        dataset = load_dataset()
    else:
        from crime.downloader.study import ensure_study_inputs
        data_release = ensure_study_inputs(
            INPUTS_DIR, mode=DATA_MODE, base_url=DATA_RELEASE_URL or None, auto_update=AUTO_UPDATE_DATA)
        display({"data_profile": DATA_PROFILE, "data_release": data_release.version,
                 "directory": str(data_release.directory)})
        if DATA_PROFILE == "hybrid":
            from crime.downloader.live import download_live_inputs
            live_data = download_live_inputs(INPUTS_DIR, source_urls=live_sources, overwrite=True, require_all=False)
            display({"live_refresh": live_data.retrieved_at, "sources": list(live_data.sources)})
        dataset = generate_dataset()
    import json
    from dataclasses import asdict
    display({"reports": len(dataset.bulletins), "tracts": len(dataset.tracts), "addresses": len(dataset.poi)})
    if dataset.reports:
        import pandas as pd
        display(pd.DataFrame([asdict(r) for r in dataset.reports]))
    (RUN_DIR / "data_sources.json").write_text(json.dumps([asdict(s) for s in dataset.sources], indent=2, default=str))
else:
    print("Municipal aggregate source selected: tract-level spatial analyses are unavailable.")


## 4. Warehouse
`auto` starts an isolated temporary PostGIS server in Colab and uses your configured
external database locally. `none` keeps exploration in memory. Temporary setup
installs PostgreSQL/PostGIS in the Colab runtime; it does not expose a public port.

Existing study tables are preserved unless **REPLACE_EXISTING_WAREHOUSE** is
selected. A reused warehouse is an existing release; this notebook's calculations
use the selected in-memory dataset, whose provenance is saved in this run.
The warehouse stores the treated SSP study tables and never replaces bulletin facts.


In [ ]:
# @title Build or connect to PostGIS
WAREHOUSE_MODE = "auto" # @param ["auto", "temporary_colab", "external", "none"]
REPLACE_EXISTING_WAREHOUSE = False # @param {type:"boolean"}
PG_HOST = "localhost" # @param {type:"string"}
PG_PORT = 5433 # @param {type:"integer"}
PG_DATABASE = "crime_db" # @param {type:"string"}
PG_USER = "admin" # @param {type:"string"}

warehouse_config = None
mode = ("temporary_colab" if IN_COLAB else "external") if WAREHOUSE_MODE == "auto" else WAREHOUSE_MODE
if mode != "none":
    from sqlalchemy import text

    from crime.config import WarehouseConfig, engine
    from crime.warehouse.notebook import load_notebook_warehouse, start_colab_postgis
    if mode == "temporary_colab":
        warehouse_config = start_colab_postgis()
    else:
        from getpass import getpass
        # Password input is hidden and does not become a saved form parameter.
        password = os.environ.get("PGPASSWORD") or os.environ.get("POSTGRES_PASSWORD") or getpass("PostGIS password: ")
        warehouse_config = WarehouseConfig(host=PG_HOST, port=PG_PORT, database=PG_DATABASE, user=PG_USER, password=password)
        del password
    db_engine = engine(warehouse_config)
    try:
        with db_engine.connect() as connection:
            existing = connection.execute(text("SELECT to_regclass('public.build') IS NOT NULL")).scalar_one()
        if dataset is not None:
            if not existing or REPLACE_EXISTING_WAREHOUSE:
                loaded = load_notebook_warehouse(dataset, warehouse_config, replace_existing=REPLACE_EXISTING_WAREHOUSE)
                display(loaded.extra["funnel"])
            else:
                print("Existing warehouse preserved; notebook analyses use the selected dataset.")
            from crime.warehouse import Warehouse
            with Warehouse(warehouse_config) as wh:
                display(wh.build())
    finally:
        db_engine.dispose()
else:
    print("In-memory mode: no database is created or modified.")


## 5. Full study workflow
The published study uses 999 global and 499 local permutations, seed 20221,
queen/rook/KNN5 weights and BH at 5%. You can increase permutations here;
the configuration and source hashes are saved with each new run. Results with
changed parameters are new analyses and may differ from the current manuscript.

Static BH families are separate by statistic, offence group and weights. EHSA
BH covers every tract-period bin in a group/frequency cube. EHSA tract-level
Mann–Kendall tests remain unadjusted: bin correction does not establish 5% FDR
for final EHSA categories. Sexual offences enter temporal analysis only.

The workflow includes seasonal MK and monthly/quarterly EHSA sensitivity when
enabled; it does not claim all possible robustness checks have been performed.


In [ ]:
# @title Analysis parameters and full pipeline
GLOBAL_PERMUTATIONS = 999 # @param {type:"integer"}
LOCAL_PERMUTATIONS = 499 # @param {type:"integer"}
SEED = 20221 # @param {type:"integer"}
MONTHLY_EHSA_SENSITIVITY = True # @param {type:"boolean"}
RUN_FULL_ANALYSIS = True # @param {type:"boolean"}
import pandas as pd

from crime.generator.workflow import WorkflowConfig, run_paper_workflow

analysis = None
if RUN_FULL_ANALYSIS:
    if dataset is not None:
        analysis = run_paper_workflow(dataset, RUN_DIR / "analysis", WorkflowConfig(
            global_permutations=GLOBAL_PERMUTATIONS, local_permutations=LOCAL_PERMUTATIONS,
            seed=SEED, monthly_ehsa=MONTHLY_EHSA_SENSITIVITY))
        for name, table in analysis.tables.items():
            display(Markdown("**" + name.replace("_", " ") + "**"))
            display(table)
    else:
        print("Run the SSP dataset preparation step first.")


## 6. Interactive space–time cube
Select a mapped offence group. The map and chart share a selected quarter;
click a tract to inspect its count history and significance before and after BH.
White means inference unavailable, not nonsignificant. Colours use fixed scales
across quarters. The exported HTML needs internet access for Leaflet and Plotly.
This viewer does not send data to a Python callback server.


In [ ]:
# @title Inspect analyzed cube histories
CUBE_GROUP = "all" # @param ["all", "property", "bodily_harm", "traffic", "drug", "violent"]
from crime.exporter.interactive import export_cube_html
from crime.visualizer.cube import cube_explorer

if analysis is not None:
    result = analysis.ehsa[CUBE_GROUP]
    viewer = cube_explorer(analysis.cubes[CUBE_GROUP], dataset.tracts, bins=result.bins, labels=result.labels)
    display(viewer)
    print(export_cube_html(viewer, RUN_DIR / ("cube_" + CUBE_GROUP + ".html")))
else:
    print("Run the report-level spatial workflow first to inspect significance histories.")


## 8. Render paper figures and Tables 6–10
Statistics are read from this run's saved outputs. Rendering does not recompute
them. With a local JBCS manuscript directory, generate and validate its complete
publication bundle. Otherwise render a portable figure/table collection; supply
Brazil outlines for the study-area map. Missing optional inputs are reported.

Publishing into the paper directory is a separate opt-in action. In Colab,
mount/upload the manuscript if you want its references checked; a local path on
your computer is not accessible from the remote runtime.


In [ ]:
# @title Publication output
RENDER_PAPER_ASSETS = True # @param {type:"boolean"}
PAPER_DIRECTORY = "" # @param {type:"string"}
BRAZIL_OUTLINES = "" # @param {type:"string"}
INCLUDE_BASEMAP = False # @param {type:"boolean"}
PUBLISH_TO_PAPER = False # @param {type:"boolean"}
if analysis is not None and RENDER_PAPER_ASSETS:
    from crime.exporter import JBCSProfile, generate_jbcs, publish_jbcs, validate_manifest
    from crime.visualizer.overview import render_overview
    from crime.visualizer.paper import RenderContext, render_paper
    persisted = RUN_DIR / "dataset"
    persisted.mkdir(exist_ok=True)
    dataset.tracts.to_parquet(persisted / "tracts.parquet", index=False)
    dataset.bulletins.to_parquet(persisted / "bulletins.parquet", index=False)
    output = RUN_DIR / "publication"
    brazil_path = Path(BRAZIL_OUTLINES).expanduser() if BRAZIL_OUTLINES else None
    if PAPER_DIRECTORY:
        profile = JBCSProfile(Path(PAPER_DIRECTORY).expanduser())
        manifest = generate_jbcs(output, profile=profile, input_dir=analysis.output_dir,
                                 tracts_path=persisted / "tracts.parquet", bulletins_path=persisted / "bulletins.parquet",
                                 brazil_path=brazil_path, basemap=INCLUDE_BASEMAP)
        validate_manifest(manifest, require_complete=True)
        display({"validated_manifest": str(manifest)})
        if PUBLISH_TO_PAPER:
            print("Published assets:", len(publish_jbcs(manifest, profile=profile)))
    else:
        if PUBLISH_TO_PAPER:
            raise ValueError("Set PAPER_DIRECTORY before publishing")
        context = RenderContext(analysis.output_dir, output, dataset.tracts, basemap=INCLUDE_BASEMAP)
        render_paper(context)
        gaps = render_overview(context, brazil_path=brazil_path, bulletins_path=persisted / "bulletins.parquet")
        display({"assets": len(context.assets), "optional_gaps": gaps, "output": str(output)})
    from IPython.display import Image
    for name in ["fig_gistar_all.png", "fig_lisa_all.png", "fig_ehsa_all.png"]:
        if (output / name).is_file():
            display(Image(filename=str(output / name), width=1000))
    for path in sorted(output.glob("tab_*.tex")):
        display(Markdown("**" + path.name + "**"))
        print(path.read_text())
else:
    print("Publication rendering requires the report-level workflow and saved spatial results.")


## Save this run
The archive contains results, reference snapshots, query provenance, and any
interactive viewer or publication assets generated above. It does not include
Google tokens or database credentials. For a full workflow it may include the
persisted treated study dataset used for publication; choose where you store it.


In [ ]:
# @title Archive results
CREATE_ARCHIVE = True # @param {type:"boolean"}
DOWNLOAD_ARCHIVE = False # @param {type:"boolean"}
if CREATE_ARCHIVE:
    import shutil
    archive = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
    print("Saved:", archive)
    if DOWNLOAD_ARCHIVE and IN_COLAB:
        from google.colab import files
        files.download(archive)
